# 1. Library calling

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime
import warnings
from tqdm import tqdm
import numpy as np
from IPython.display import clear_output
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")
import os

# 2. Defining the Product Information and Location

In [2]:
Links=["https://www.amazon.com/dp/B000ALG02Q",
"https://www.amazon.com/dp/B000DN7Q3S",
"https://www.amazon.com/dp/B001DCN1OG",
"https://www.amazon.com/dp/B003O0Y0BI",
"https://www.amazon.com/dp/B000ALBRQ0",
"https://www.amazon.com/dp/B000ALG0KS",
"https://www.amazon.com/dp/B000ALIS0I",
"https://www.amazon.com/dp/B000JZSPBM",
"https://www.amazon.com/dp/B000DINHFO",
"https://www.amazon.com/dp/B000DT7SSK",
"https://www.amazon.com/dp/B0D1KV5S77",
"https://www.amazon.com/dp/B000GBGNU4",
"https://www.amazon.com/dp/B000GBBP20",
"https://www.amazon.com/dp/B000JZWRNE",
"https://www.amazon.com/dp/B000JZSPH6",
"https://www.amazon.com/dp/B0031HQ0PI",
"https://www.amazon.com/dp/B001DCN1SC",
"https://www.amazon.com/dp/B00PY5K56U",
"https://www.amazon.com/dp/B003NHEPEE",
"https://www.amazon.com/dp/B006ZSUKXI",
"https://www.amazon.com/dp/B000ALIS3A",
"https://www.amazon.com/dp/B003O1AI22",
"https://www.amazon.com/dp/B001DCN1VE",
"https://www.amazon.com/dp/B0031HNVTG",
"https://www.amazon.com/dp/B000ALBS0K"
]

In [3]:
length=len(Links)
print(length)

25


# 3. Setting Webdriver and Website Specific Information

In [4]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
wait=WebDriverWait(driver, 5)
driver.get('https://www.amazon.com/')


In [5]:
location = driver.find_element(By.ID, 'nav-global-location-popover-link')

location.click()
sleep(3)
pin = driver.find_element(By.ID, 'GLUXZipUpdateInput')
pinnumber='94203'
pin.send_keys(pinnumber)

apply=driver.find_element(By.CSS_SELECTOR, 'span.a-button-inner[data-action="GLUXPostalUpdateAction"]')
apply.click()


In [6]:
driver.find_element(By.CSS_SELECTOR,'[data-csa-c-type="link"]').click()

In [7]:
eid=driver.find_element(By.CSS_SELECTOR,'[id="ap_email"]')
eid.click()
emailid="Vikram.Vadhirajan@firstbrandsgroup.com"
eid.send_keys(emailid)
eid.send_keys(Keys.ENTER)
sleep(2)

passw=driver.find_element(By.CSS_SELECTOR,'[id="ap_password"]')
passw.click()
password="FBG@Nov-2024"
passw.send_keys(password)
passw.send_keys(Keys.ENTER)

# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [8]:
cols =['ASIN'] #,'Review_Mentions','Standards'
df = pd.DataFrame(columns=cols)
df
count=0

In [9]:
driver.get(Links[2])

In [10]:
len(Links)

25

In [11]:
for i in tqdm(range (len(Links))):
    print(f"Number of Items scanned:{i}")
    driver.get(Links[i])
    sleep(2)
    try:
        Name=driver.find_element(By.ID,"title").text
        try:
            driver.find_element(By.CSS_SELECTOR,'[id="acrCustomerReviewText"]').click()
            sleep(3)
            driver.find_element(By.CSS_SELECTOR,'[data-hook="see-all-reviews-link-foot"]').click()
            sleep(1)
            var=0
            driver.find_element(By.CLASS_NAME,'cr-sort-dropdown').click()
            driver.find_element(By.CSS_SELECTOR,'[aria-labelledby="sort-order-dropdown_1"]').click()
            sleep(1)
            for s in range(5):
                driver.find_element(By.CLASS_NAME,'star-rating-select').click()
                driver.find_element(By.ID,f"star-count-dropdown_{s+1}").click()
                sleep(1)
                for m in range(100):
                    sleep(2)
                    #print("1")
                    rbs=driver.find_elements(By.CSS_SELECTOR,'[data-hook="review-body"]')
                    titles=driver.find_element(By.CSS_SELECTOR,'[id="cm_cr-review_list"]').find_elements(By.CSS_SELECTOR,'[data-hook="review-title"]')
                    starnum=driver.find_element(By.CSS_SELECTOR,'[id="reviews-filter-info-segment"]').text
                    revdates=driver.find_elements(By.CSS_SELECTOR,'[data-hook="review-date"]')
                    variants=driver.find_elements(By.CSS_SELECTOR,'[data-hook="format-strip"]')
                    print(len(variants))
                    if len(variants)==0:                    
                        for rb, title,revdate in zip(rbs,titles,revdates):
                            df.loc[count,"#Comments"]=var+1
                            df.loc[count,"ASIN"]=Links[i].split("/dp/")[1]
                            df.loc[count, "Title"]=title.text
                            df.loc[count, "Comments"]=rb.text
                            df.loc[count,'Review Date']=revdate.text.split(" on ")[-1]
                            df.loc[count,'Star']=starnum
                            df.loc[count,"Name"]=Name
                            try:
                                df.loc[count,"Variant"]=variant.text
                            except:
                                df.loc[count,"Variant"]=""
                            count=count+1
                            var=var+1
                            clear_output(wait=True)
                        try:
                            driver.find_element(By.CSS_SELECTOR,'[class="a-last"]').click()
                        except:
                            break
                    else:
                        for rb, title,revdate,variant in zip(rbs,titles,revdates,variants):
                            df.loc[count,"#Comments"]=var+1
                            df.loc[count,"ASIN"]=Links[i].split("/dp/")[1]
                            df.loc[count, "Title"]=title.text
                            df.loc[count, "Comments"]=rb.text
                            df.loc[count,'Review Date']=revdate.text.split(" on ")[-1]
                            df.loc[count,'Star']=starnum
                            df.loc[count,"Name"]=Name
                            try:
                                df.loc[count,"Variant"]=variant.text
                            except:
                                df.loc[count,"Variant"]=""
                            count=count+1
                            var=var+1
                            clear_output(wait=True)
                        try:
                            driver.find_element(By.CSS_SELECTOR,'[class="a-last"]').click()
                        except:
                            break
            print(f"Number of comments:{var}")   
        except:
            pass
    except:
        pass


100%|██████████| 25/25 [16:31<00:00, 39.66s/it]

Number of comments:170


In [12]:
df

,ASIN,#Comments,Title,Comments,Review Date,Star,Name,Variant
0,B000ALG02Q,1.0,Works Great!,1996 Ford Explorer XLT - These struts are perf...,"April 14, 2023",5 star,StrongArm 4026 Ford Explorer and Mercury Mount...,Size: Pair Pack of 2
1,B000ALG02Q,2.0,It’s now safe working under my hood with Stron...,Installed these on the hood and they work grea...,"March 23, 2023",5 star,StrongArm 4026 Ford Explorer and Mercury Mount...,Size: Pair Pack of 2
2,B000ALG02Q,3.0,These are everything I needed for my heavy For...,I wasn't able to install these myself because ...,"November 19, 2022",5 star,StrongArm 4026 Ford Explorer and Mercury Mount...,Size: Pair Pack of 2
3,B000ALG02Q,4.0,Great product,Good,"January 21, 2021",5 star,StrongArm 4026 Ford Explorer and Mercury Mount...,Size: Pair Pack of 2
4,B000ALG02Q,5.0,holds great,"holds the hood up great, no more stick holder","November 24, 2020",5 star,StrongArm 4026 Ford Explorer and Mercury Mount...,Size: Pair Pack of 2
...,...,...,...,...,...,...,...,...
1881,B000ALBS0K,166.0,PIEACE OF JUNK BOLT,The bolts securing the lift support arm on my ...,"January 25, 2014",1 star,"StrongArm 4699 Liftgate / Hatch Lift Support, ...",Size: Pack of 1
1882,B000ALBS0K,167.0,not fit,"The product does not fit in the car, I have a ...","September 23, 2013",1 star,"StrongArm 4699 Liftgate / Hatch Lift Support, ...",Size: Pack of 1
1883,B000ALBS0K,168.0,Horrible!,I looked this unit up to make sure that it wou...,"August 19, 2013",1 star,"StrongArm 4699 Liftgate / Hatch Lift Support, ...",Size: Pack of 1
1884,B000ALBS0K,169.0,"After 1 1/2 years, it no longer is able to kee...","After 1 1/2 years, these supports are no longe...","April 20, 2013",1 star,"StrongArm 4699 Liftgate / Hatch Lift Support, ...",Size: Pack of 1


In [13]:
OFolder=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Data Scraping - Amazon data\Amazon"
from datetime import date
Date=str(date.today()).replace("-","")
Source="Amazon_"
search_text="Strong_Arm_Lift_Support"
with pd.ExcelWriter(OFolder+'\\'+f'{Date}_{Source}ProductReviews_'+search_text+'.xlsx') as writer:  # doctest: +SKIP
    df.to_excel(writer,index=False, sheet_name='Reviews')

In [79]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [114]:
%pip install nltk textblob

Note: you may need to restart the kernel to use updated packages.


In [116]:
import nltk
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Vikram.Vadhirajan\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Vikram.Vadhirajan\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Vikram.Vadhirajan\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from textblob import TextBlob
nltk.download('vader_lexicon')

In [82]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Initialize VADER sentiment analyzer
sid = SentimentIntensityAnalyzer()


In [107]:
cols =["Positive Words","Negative Words"] #,'Review_Mentions','Standards'
dfs = pd.DataFrame(columns=cols)
dfs
count=0

In [117]:
for i in tqdm(range(len(df))):
    # Sample text
    text =df["Comments"][i]
    # Tokenize the text into words (or word chunks)
    words = word_tokenize(text)

    # Load stop words
    stop_words = set(stopwords.words('english'))

    # Remove stop words
    filtered_words = [word for word in words if word.lower() not in stop_words]

    # Analyze the sentiment for each word
    word_sentiments = {}
    for word in filtered_words:
        word_sentiment = sid.polarity_scores(word)
        word_sentiments[word] = word_sentiment['compound']

    # # Display each word with its sentiment score
    # for word, score in word_sentiments.items():
    #     print(f"Word: '{word}', Sentiment Score: {score}")

    # Interpretation based on each word's compound score
    PW=[word for word, score in word_sentiments.items() if score >= 0.05]
    NW=[word for word, score in word_sentiments.items() if score <= -0.05]
    dfs.at[i,"Comments"] = text
    dfs.at[i,"ASIN"] = df["ASIN"][i]
    dfs.at[i,"Positive Words"] = PW
    dfs.at[i,"Negative Words"] = NW

    # print("\nPositive words:", positive_words)
    # print("Negative words:", negative_words)


100%|██████████| 786/786 [00:01<00:00, 592.84it/s]


In [118]:
dfs

,Positive Words,Negative Words,Comments,ASIN
0,"[perfect, easy, well, prevents]",[],perfect for my needs.\neasy to use.\nthe tight...,B0B2T3F3HM
1,[good],[],Very heavy duty everything about it is good,B0B2T3F3HM
2,"[enjoyed, secure, solid]",[],Really enjoyed using this with a cargo carrier...,B0B2T3F3HM
3,"[perfect, solution, secure, want]","[worried, exhaust, damage]",I wanted to add a small cargo carrier to our c...,B0B2T3F3HM
4,"[great, good, like, want, dedicated]",[bad],Using with a Curt spare tire carrier. Works gr...,B0B2T3F3HM
...,...,...,...,...
781,"[easily, best, easy]",[sucker],I have owned several hitch racks and this is e...,B09VCNQS75
782,"[like, happier, play, nice, peace, definitely]",[worry],This thing is a tank. More than what I reallly...,B09VCNQS75
783,"[Play, impression, help, natural, pretty, conf...","[mess, problem, shake, disappointment]",Play Video\nDon't let the shiny blue anodized ...,B09VCNQS75
784,"[like, good, well]","[sucks, bad, low]",i like the welds also good heavy stock it went...,B07QMP6W8B


# 5. Post Processing Data and Exporting

In [120]:
with pd.ExcelWriter(OFolder+'\\'+f'{Source}ProductSentiment_'+search_text+'.xlsx') as writer:  # doctest: +SKIP
    df.to_excel(writer,index=False, sheet_name='Comments')
    dfs.to_excel(writer,index=False, sheet_name='Sentiment')

# 99. Archived Codes